# Irrigated vs rainfed cropland — AlphaEarth Foundations embeddings

**Goal.** Measure how well the AlphaEarth Foundations (AEF) annual embeddings separate irrigated from rainfed parcels in Morocco, under the spatial evaluation protocol shared by all models in this study.

**Data.** AEF 2024 annual embeddings: 64 dimensions per 10 m pixel, read as a cloud-native Zarr mosaic from [Source Cooperative](https://source.coop/tge-labs/aef-mosaic) with `xarray` (no Earth Engine). Reference: [auspatious cloud-native EO workshop](https://github.com/auspatious/cloud-native-geospatial-eo-workshop/blob/main/04_Alpha_Earth_Embeddings.ipynb).

**Workflow**
1. Load the labelled parcels and spatial folds
2. Extract the mean embedding of every parcel, block by block, with a resumable cache
3. Check feature quality and class separability
4. Tune, test and cross-validate three classifiers (shared protocol in `irrmap/evaluation.py`)
5. Feature importance
6. Save results to `outputs/models/alphaearth_2024/`

Mapping (wall-to-wall prediction) is left for a later notebook.

In [ ]:
import json
from pathlib import Path

import geopandas as gpd
import joblib
import matplotlib.pyplot as plt
import numpy as np
import odc.geo.xr  # noqa: F401  (registers the .odc accessor)
import pandas as pd
import xarray as xr
from odc.geo.geom import box
from sklearn.decomposition import PCA
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import make_scorer, f1_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from irrmap.evaluation import (HEADLINE, block_bootstrap, best_params, cross_validate, plot_confusion,
                               plot_cv_comparison, predict, tune_on_validation)
from irrmap.features import block_bbox, extract_by_block, parcel_zonal_mean

pd.set_option("display.width", 200)

In [ ]:
ROOT = Path.cwd()
SPLIT_FILE = ROOT / "outputs" / "split" / "parcels_split.geojson"

YEAR = 2024
RUN_NAME = f"alphaearth_{YEAR}"
FEATURE_FILE = ROOT / "outputs" / "features" / f"{RUN_NAME}.parquet"
CACHE_DIR = ROOT / "outputs" / "features" / "_cache" / RUN_NAME          # one parquet per 20 km block
MODEL_DIR = ROOT / "outputs" / "models" / RUN_NAME
MODEL_DIR.mkdir(parents=True, exist_ok=True)

AEF_ZARR = "s3://us-west-2.opendata.source.coop/tge-labs/aef-mosaic/"
AEF_NODATA = -128                                                        # int8 fill value of the mosaic
AEF_BANDS = [f"A{i:02d}" for i in range(64)]

TEST_FOLD, VAL_FOLD = 0, 1
SEED = 42
N_BOOT = 1000

## 1. Reference parcels and spatial folds

Parcels and folds come from `2_split_train_test.ipynb`: 20 km blocks in 5 folds, fold 0 = test, fold 1 = validation, folds 2–4 = train.

In [ ]:
parcels = gpd.read_file(SPLIT_FILE)
parcels = parcels[["parcel_id", "label", "class_name", "block_id", "fold", "random_fold", "split", "geometry"]]
pd.crosstab(parcels["split"], parcels["class_name"], margins=True).reindex(["train", "val", "test", "All"])

class_name,Irrigated,Rainfed,All
split,,,
train,4529,718,5247
val,1528,239,1767
test,1528,239,1767
All,7585,1196,8781


## 2. Parcel embeddings

The mosaic is opened lazily, so only metadata is read until a window is computed. For each 20 km block the window around its parcels is loaded, the int8 values are de-quantised to floats, and the 64 dimensions are averaged over the pixels inside each parcel (shrunk by 5 m to avoid edge pixels).

De-quantisation follows the AEF convention: `value = (q / 127.5)² · sign(q)`, with −128 as no-data.

The result is cached per block; delete `outputs/features/` to rebuild.

In [ ]:
def open_aef() -> xr.Dataset:
    """Lazy handle on the global AEF mosaic (dims: time, band, y, x)."""
    try:
        import s3fs  # noqa: F401  (anonymous S3 access)
        import zarr
    except ImportError as err:
        raise ImportError("Reading the AEF mosaic needs zarr>=3 and s3fs: "
                          "conda install -c conda-forge 'zarr>=3' s3fs") from err
    if int(zarr.__version__.split(".")[0]) < 3:
        raise ImportError(f"zarr {zarr.__version__} found; the AEF mosaic is Zarr v3, install zarr>=3")
    ds = xr.open_zarr(AEF_ZARR, storage_options={"anon": True}, consolidated=False, chunks="auto", zarr_format=3)
    return ds.odc.assign_crs("EPSG:4326")


def aef_window(aef: xr.Dataset, bbox) -> xr.DataArray:
    """De-quantised embeddings for one bbox and year, dims (feature, y, x), NaN = no-data."""
    q = odc.geo.xr.crop(aef, box(*bbox, crs="EPSG:4326")).sel(time=YEAR)["embeddings"]
    q = q.transpose("band", ...).compute()
    valid = (q != AEF_NODATA).any("band")
    emb = ((q.astype("float32") / 127.5) ** 2) * np.sign(q)
    return emb.where(valid).rename(band="feature").assign_coords(feature=AEF_BANDS)


def aef_block_features(group: gpd.GeoDataFrame) -> pd.DataFrame:
    return parcel_zonal_mean(aef_window(AEF, block_bbox(group)), group)

In [ ]:
if FEATURE_FILE.exists():
    features = pd.read_parquet(FEATURE_FILE)
    print(f"Loaded cached features: {features.shape}")
else:
    AEF = open_aef()
    display(AEF)
    features, failed = extract_by_block(parcels, aef_block_features, CACHE_DIR)
    if failed:
        print(f"{len(failed)} blocks failed — re-run this cell to retry them: {failed}")
    else:
        FEATURE_FILE.parent.mkdir(parents=True, exist_ok=True)
        features.to_parquet(FEATURE_FILE)
        print(f"Saved {FEATURE_FILE.relative_to(ROOT)}: {features.shape}")

/opt/anaconda3/envs/gdal/lib/python3.13/site-packages/zarr/core/group.py:3609: ZarrUserWarning: Object at .checkpoint is not recognized as a component of a Zarr hierarchy.
  warnings.warn(
/opt/anaconda3/envs/gdal/lib/python3.13/site-packages/zarr/core/group.py:3609: ZarrUserWarning: Object at README.md is not recognized as a component of a Zarr hierarchy.
  warnings.warn(


<xarray.Dataset> Size: 4PB
Dimensions:      (time: 9, band: 64, y: 1859584, x: 4009984)
Coordinates:
  * time         (time) int32 36B 2017 2018 2019 2020 2021 2022 2023 2024 2025
  * x            (x) float64 32MB -180.0 -180.0 -180.0 ... 180.2 180.2 180.2
  * band         (band) object 512B 'A00' 'A01' 'A02' ... 'A61' 'A62' 'A63'
  * y            (y) float64 15MB 83.69 83.69 83.69 ... -83.36 -83.36 -83.36
    spatial_ref  int32 4B 4326
Data variables:
    embeddings   (time, band, y, x) int8 4PB dask.array<chunksize=(1, 64, 256, 256), meta=np.ndarray>
Attributes: (12/15)
    proj:code:               EPSG:4326
    spatial:dimensions:      ['y', 'x']
    spatial:transform:       [8.983111749910169e-05, 0.0, -180.0, 0.0, -8.983...
    spatial:transform_type:  affine
    spatial:bbox:            [-180.0, -83.36280346631479, 180.2213438735178, ...
    spatial:shape:           [1859584, 4009984]
    ...                      ...
    geoemb:model:            https://developers.google.com/earth-engine/datas...
    geoemb:source_data:      https://source.coop/tge-labs/aef/v1/annual/
    geoemb:data_type:        int8
    geoemb:gsd:              8.983111749910169e-05
    geoemb:quantization:     {'method': 'signed_square', 'original_dtype': 'f...
    zarr_conventions:        [{'uuid': 'f17cb550-5864-4468-aeb7-f3180cfb622f'...

blocks:   0%|          | 0/91 [00:00<?, ?it/s]

## 3. Feature checks

In [ ]:
FEATURES = AEF_BANDS
data = (parcels.drop(columns="geometry").set_index("parcel_id")
               .join(features, how="inner"))
no_signal = data[FEATURES].isna().all(axis=1)
print(f"Parcels with features: {len(data):,} of {len(parcels):,}; without any valid pixel: {no_signal.sum()}")
data = data[~no_signal]
print("Pixels per parcel:", data["n_px"].describe(percentiles=[0.05, 0.5, 0.95]).round(0).to_dict())

In [ ]:
# Two-component PCA of the parcel embeddings: a quick look at how separable the classes are before any training.
pcs = PCA(n_components=2, random_state=SEED).fit(data[FEATURES])
xy = pcs.transform(data[FEATURES])
fig, ax = plt.subplots(figsize=(6.5, 5.5), dpi=120)
for lab, name, color in [(1, "Irrigated", "#1f77b4"), (2, "Rainfed", "#d62728")]:
    m = data["label"].to_numpy() == lab
    ax.scatter(xy[m, 0], xy[m, 1], s=4, alpha=0.4, color=color, label=f"{name} ({m.sum():,})")
ax.set_xlabel(f"PC1 ({pcs.explained_variance_ratio_[0]:.0%})")
ax.set_ylabel(f"PC2 ({pcs.explained_variance_ratio_[1]:.0%})")
ax.set_title("Parcel embeddings, first two principal components", fontweight="bold")
ax.legend(frameon=False, markerscale=3)
fig.tight_layout()
fig.savefig(MODEL_DIR / "pca_parcels.png", dpi=150, bbox_inches="tight")
plt.show()

## 4. Modelling protocol

| Step | Train on | Evaluate on | Purpose |
|---|---|---|---|
| Tuning | folds 2–4 | fold 1 (validation) | choose hyper-parameters by macro-F1 |
| Final test | folds 1–4 | fold 0 (test) | headline scores, block-bootstrap 95% CI |
| Spatial 5-fold CV | 4 folds | the 5th | stability across areas (mean ± std) |
| Random 5-fold CV | 4 folds | the 5th | same protocol on a random split, to show how much it overstates accuracy |

Three classifiers span the usual range:

- **Logistic regression**: a *linear probe*, the standard check of how much class information an embedding holds without a complex decoder.
- **Random forest**: the conventional baseline of the proposal.
- **Histogram gradient boosting**: a strong tabular learner that handles missing values natively.

Macro-F1 is the selection metric: with 14% rainfed parcels, overall accuracy rewards ignoring the minority class.

In [ ]:
def build_logreg(C=1.0):
    return make_pipeline(SimpleImputer(strategy="median"), StandardScaler(),
                         LogisticRegression(C=C, class_weight="balanced", max_iter=5000))

def build_rf(max_features="sqrt", min_samples_leaf=1, class_weight=None):
    return make_pipeline(SimpleImputer(strategy="median"),
                         RandomForestClassifier(n_estimators=500, max_features=max_features,
                                                min_samples_leaf=min_samples_leaf, class_weight=class_weight,
                                                n_jobs=-1, random_state=SEED))

def build_hgb(learning_rate=0.1, max_leaf_nodes=31, class_weight=None):
    # Early stopping would carve a random (non-spatial) validation split, so it is switched off.
    return HistGradientBoostingClassifier(learning_rate=learning_rate, max_leaf_nodes=max_leaf_nodes,
                                          max_iter=300, early_stopping=False, class_weight=class_weight,
                                          random_state=SEED)

MODELS = {
    "logreg": (build_logreg, {"C": [0.01, 0.1, 1.0, 10.0]}),
    "rf": (build_rf, {"max_features": ["sqrt", 0.3], "min_samples_leaf": [1, 5],
                      "class_weight": [None, "balanced_subsample"]}),
    "hgb": (build_hgb, {"learning_rate": [0.05, 0.1], "max_leaf_nodes": [15, 31],
                        "class_weight": [None, "balanced"]}),
}

In [ ]:
X = data[FEATURES]
y = data["label"]
fold = data["fold"]
is_test = fold.eq(TEST_FOLD).to_numpy()
is_val = fold.eq(VAL_FOLD).to_numpy()
is_train = ~(is_test | is_val)
print(f"{len(FEATURES)} features | train {is_train.sum():,} · validation {is_val.sum():,} · test {is_test.sum():,}")

### 4.1 Hyper-parameter tuning on the validation fold

In [ ]:
tuning = tune_on_validation(MODELS, X, y, is_train, is_val)
BEST = best_params(tuning)

view = tuning.assign(params=tuning["params"].astype(str))[["model", "params", "macro_f1", "f1_rainfed", "f1_irrigated", "balanced_accuracy"]]
display(view.groupby("model").head(3).round(3).set_index(["model", "params"]))
print("Selected:", BEST)

### 4.2 Final test (fold 0)

Each model is refitted with its selected parameters on folds 1–4 and scored once on the held-out test fold. Confidence intervals resample whole 20 km blocks, because parcels within a block are not independent.

In [ ]:
fitted, test_pred, test_ci = {}, {}, {}
for name, (build, _) in MODELS.items():
    model = build(**BEST[name]).fit(X[~is_test], y[~is_test])
    y_hat, p = predict(model, X[is_test])
    fitted[name] = model
    test_pred[name] = pd.DataFrame({"y_true": y[is_test], "y_pred": y_hat, "p_rainfed": p,
                                    "block_id": data.loc[is_test, "block_id"]})
    test_ci[name] = block_bootstrap(y[is_test], y_hat, p, data.loc[is_test, "block_id"], n_boot=N_BOOT, seed=SEED)

def fmt(ci):
    return ci.apply(lambda r: f"{r.estimate:.3f} [{r.ci_low:.3f}–{r.ci_high:.3f}]", axis=1)

test_table = pd.DataFrame({m: fmt(ci) for m, ci in test_ci.items()}).loc[HEADLINE]
test_table

In [ ]:
fig, axes = plt.subplots(1, len(MODELS), figsize=(4.6 * len(MODELS), 4.2), dpi=120)
for ax, (name, pred) in zip(np.atleast_1d(axes), test_pred.items()):
    plot_confusion(pred["y_true"], pred["y_pred"], ax, title=f"{name} — macro-F1 {test_ci[name].loc['macro_f1', 'estimate']:.3f}")
fig.suptitle(f"{RUN_NAME}: test fold (n = {is_test.sum():,})", fontweight="bold")
fig.tight_layout()
fig.savefig(MODEL_DIR / "test_confusion.png", dpi=150, bbox_inches="tight")
plt.show()

### 4.3 Spatial vs random cross-validation

Every fold is held out once. The tuned parameters are reused unchanged; they were chosen on fold 1, so fold 1's spatial score is slightly optimistic.

In [ ]:
cv_rows = []
for name, (build, _) in MODELS.items():
    for scheme, col in [("spatial", "fold"), ("random", "random_fold")]:
        per_fold, _ = cross_validate(lambda: build(**BEST[name]), X, y, data[col])
        cv_rows.append(per_fold.reset_index().assign(model=name, cv=scheme))
cv = pd.concat(cv_rows, ignore_index=True)

cv_summary = (cv.groupby(["model", "cv"])[["macro_f1", "f1_rainfed", "f1_irrigated", "overall_accuracy"]]
                .agg(["mean", "std"]).round(3))
gap = (cv.groupby(["model", "cv"])["macro_f1"].mean().unstack()
         .assign(optimism=lambda t: t["random"] - t["spatial"]).round(3))
display(cv_summary)
gap.rename(columns={"optimism": "random − spatial (macro-F1)"})

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4), dpi=120)
plot_cv_comparison(cv, "macro_f1", axes[0])
plot_cv_comparison(cv, "f1_rainfed", axes[1])
axes[0].set_title("Macro-F1 per fold", fontweight="bold")
axes[1].set_title("Rainfed F1 per fold", fontweight="bold")
fig.suptitle(f"{RUN_NAME}: random vs spatial 5-fold CV", fontweight="bold")
fig.tight_layout()
fig.savefig(MODEL_DIR / "cv_random_vs_spatial.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. Which embedding dimensions matter

Permutation importance on the test fold for the random forest: the drop in macro-F1 when one dimension is shuffled. AEF dimensions have no physical meaning, so this mainly shows whether the signal is spread or concentrated.

In [ ]:
macro_f1 = make_scorer(f1_score, average="macro")
imp = permutation_importance(fitted["rf"], X[is_test], y[is_test], scoring=macro_f1,
                             n_repeats=10, random_state=SEED, n_jobs=-1)
importance = (pd.DataFrame({"mean": imp.importances_mean, "std": imp.importances_std}, index=FEATURES)
                .sort_values("mean", ascending=False))
top = importance.head(20).iloc[::-1]
fig, ax = plt.subplots(figsize=(6, 6), dpi=120)
ax.barh(top.index, top["mean"], xerr=top["std"], color="#5b8c5a")
ax.set_xlabel("Drop in macro-F1 when shuffled")
ax.set_title("Top 20 AEF dimensions (random forest, test fold)", fontweight="bold")
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
fig.savefig(MODEL_DIR / "permutation_importance.png", dpi=150, bbox_inches="tight")
plt.show()
importance.to_csv(MODEL_DIR / "permutation_importance.csv")

## 6. Save results

Everything needed to compare runs later is written to the run folder: the tuning table, test scores with CIs, per-fold CV scores, test predictions and the fitted models.

In [ ]:
tuning.assign(params=tuning["params"].astype(str)).to_csv(MODEL_DIR / "tuning_validation.csv", index=False)
pd.concat({m: ci for m, ci in test_ci.items()}, names=["model"]).to_csv(MODEL_DIR / "test_metrics.csv")
cv.to_csv(MODEL_DIR / "cv_folds.csv", index=False)
pd.concat(test_pred, names=["model"]).to_parquet(MODEL_DIR / "test_predictions.parquet")
for name, model in fitted.items():
    joblib.dump(model, MODEL_DIR / f"model_{name}.joblib")
(MODEL_DIR / "run_config.json").write_text(json.dumps({
    "run": RUN_NAME, "year": YEAR, "n_features": len(FEATURES), "n_parcels": int(len(data)),
    "test_fold": TEST_FOLD, "val_fold": VAL_FOLD, "seed": SEED, "best_params": BEST,
}, indent=2, default=str))
print("Saved to", MODEL_DIR.relative_to(ROOT))
sorted(p.name for p in MODEL_DIR.iterdir())